# Does narrowing the network change the shortest route?

The catalog lets an experiment work with a slice of the network -- one
airline, one class of airport, one country. That is a modelling choice,
not a neutral one: removing edges can only make the shortest path longer
or leave it alone, and removing enough of them makes it disappear.

This experiment measures the size of that effect on the committed data.

The data is pinned by `experiment.toml` and verified on load. If a byte
of the snapshot changes, this notebook raises rather than quietly
producing a different number.

## Setup

Only this first cell differs between Colab and a local checkout.

In [ ]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit(
        'install the package first -- see the comment above'
    ) from error

In [ ]:
from pathlib import Path

from flight_planner.experiments import Experiment

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters
experiment

## The data

Opening the snapshot re-hashes every file against the manifest, so what
follows is known to be the data the experiment was pinned to.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria or '(full network)')

catalog = experiment.catalog()
catalog

## The narrowing

Every narrowing reports what it cost. Under the default `both` endpoint
mode the order of these three makes no difference to the result -- closed
narrowing commutes -- but the chain records the order that ran.

In [ ]:
narrowed = (
    catalog
    .airline(parameters['airline'])
    .airport_type(parameters['airport_type'])
    .country(parameters['country'])
)

summary = narrowed.summary()
for step, routes, airports in zip(
    summary['chain'], summary['routes'][1:], summary['airports'][1:]
):
    arguments = ', '.join(str(value) for value in step['arguments'])
    print(f"{step['operation']}({arguments}): {routes} routes / {airports} airports")

## The question

Dijkstra over both graphs, weighted by great-circle distance.

In [ ]:
whole = catalog.planner()
slice_ = narrowed.planner()


def shortest(planner, origin, destination):
    """Return (km, legs) for a pair, or (None, []) if unreachable."""
    try:
        distance_km, legs = planner.find_shortest_route(origin, destination)
    except ValueError:
        return None, []  # An endpoint is not in this graph at all.
    if distance_km == float('inf'):
        return None, []
    return distance_km, legs

In [ ]:
origin = parameters['origin']
destination = parameters['destination']

whole_km, whole_legs = shortest(whole, origin, destination)
slice_km, slice_legs = shortest(slice_, origin, destination)

print(f'{origin} -> {destination}')
print(f'  whole network : {whole_km:,.1f} km in {len(whole_legs)} leg(s)')
print(f'  narrowed      : {slice_km:,.1f} km in {len(slice_legs)} leg(s)')
for leg in slice_legs:
    print(f"    {leg.flight_number}  {leg.origin.city} -> {leg.destination.city}")

## Where the narrowing does bite

The headline pair is served directly by the airline under test, so the
slice costs nothing. Other pairs are not so lucky.

In [ ]:
rows = []
for pair_origin, pair_destination in parameters['comparison_pairs']:
    a_km, a_legs = shortest(whole, pair_origin, pair_destination)
    b_km, b_legs = shortest(slice_, pair_origin, pair_destination)
    rows.append({
        'pair': f'{pair_origin}-{pair_destination}',
        'whole_km': a_km,
        'whole_legs': len(a_legs),
        'narrowed_km': b_km,
        'narrowed_legs': len(b_legs),
        'extra_km': None if b_km is None else round(b_km - a_km, 1),
    })

for row in rows:
    if row['narrowed_km'] is None:
        verdict = 'unreachable'
    elif row['extra_km'] == 0:
        verdict = 'unchanged'
    else:
        verdict = f"+{row['extra_km']:,.1f} km"
    print(
        f"{row['pair']:>9}  whole {row['whole_km']:8,.1f} km /{row['whole_legs']:2d}"
        f"   narrowed {verdict}"
    )

## The answer

Recorded next to the data that produced it, and next to the narrowing
chain that derived the graph.

In [ ]:
experiment.record(
    {
        'pair': f'{origin}-{destination}',
        'whole_network_km': whole_km,
        'whole_network_legs': len(whole_legs),
        'narrowed_km': slice_km,
        'narrowed_legs': len(slice_legs),
        'narrowed_flights': [leg.flight_number for leg in slice_legs],
        'comparison': rows,
    },
    catalog=narrowed,
)

### Reading the result

`results.json` carries the snapshot id, its criteria and the commit it was
generated from, so the numbers above are attributable. Re-running against
a different snapshot changes the id in that file -- which is the point.